# Support Vector Machine (SVM)

Implementing Linear Support Vector Classification (LinearSVC) to find the optimal hyperplane for classification.

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
from metrics_utils import calculate_business_metrics


mlflow.set_experiment("Walmart-Sales-Classification")

In [ ]:
train_df = pd.read_csv('../../data/model_ready/train.csv')
test_df = pd.read_csv('../../data/model_ready/test.csv')

features_selected = [
    "Size", "Store", "Dept", "CPI", "DeptFrequency", 
    "Week_cos", "IsPreHoliday", "Week_sin", "Fuel_Price", 
    "ConsumerConfRatio", "AvgMarkDownAmount"
]
target = "Sales_Class"
holiday_col = "IsHoliday"

X_train = train_df[features_selected]
y_train = train_df[target]
X_test = test_df[features_selected]
y_test = test_df[target]
test_holidays = test_df[holiday_col]

In [ ]:
with mlflow.start_run(run_name="SVM"):
    model_path = 'svm_model.pkl'
    if os.path.exists(model_path):
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
        print('Model loaded from pickle')
    else:
        model = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
        model.fit(X_train, y_train)
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)
        print('Model trained and saved to pickle')


    y_pred = model.predict(X_test)

   
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")


    biz_metrics = calculate_business_metrics(y_test, y_pred, test_holidays)


    mlflow.log_param("model_type", "SVM")

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_weighted", f1)
    mlflow.log_metric("holiday_accuracy", biz_metrics["holiday_accuracy"])
    mlflow.log_metric("weighted_classification_error", biz_metrics["weighted_classification_error"])

    mlflow.log_artifact(model_path)

    print(f"Accuracy: {acc:.4f}")
    print(f"Holiday Accuracy: {biz_metrics['holiday_accuracy']:.4f}")
    print(f"Weighted Error: {biz_metrics['weighted_classification_error']:.4f}")
